# REINVENT4 M1: 3 seeds + 7 Case evaluators + 7 independent oracles

**Версия для Google Colab, анализа и защиты.**

Этот notebook заменяет предыдущую одно-seed/без-oracle схему. Он использует текущие модели из `suharevalexey/Case` на pinned commit `aa9c32fb26a39e518e420bb333298a20d59cbd94`.

### Научный протокол

**Guidance / reward:** 7 surrogate evaluator-моделей.  
**Independent final evaluation:** 7 oracle-моделей.  
**Random seeds:** `42`, `101`, `2024`.  
**Priorities:** 7 — по одному на каждое свойство.  
**Всего M1-агентов:** 7 × 3 = **21**.

Критический принцип:

`REINVENT -> surrogate evaluators -> Pareto/reward -> update`

а независимый oracle вызывается **только после финального sampling**. Он не может влиять на reward или обновление весов REINVENT.

## 1. Исследовательская гипотеза

**Гипотеза:** Pareto-RL с раздельными evaluator-экспертами A/B повышает независимый `JSR_Oracle` относительно базовой стратегии B0, при этом сохраняет novelty/diversity и не уводит основную массу кандидатов за пределы Applicability Domain.

Мы проверяем её на трёх независимых seeds и сообщаем результаты как **mean ± std**.

Почему это важно: высокий surrogate reward сам по себе недостаточен. Если M1 просто научился эксплуатировать ошибки evaluator-ов, рост `JSR_Surrogate` не должен устойчиво переноситься на независимые oracle-модели.

## 2. Какие модели используются

В текущем `Case/models` есть **7 evaluator joblib** и соответствующие **7 независимых oracle joblib**:

- Group A: absorption maximum, log extinction, photochemical efficiency, log half-life;
- Group B: logKp, skin sensitization, skin irritation.

Обе группы используют одно и то же представление новой молекулы: **1024-bit Morgan radius 2 + 12 RDKit descriptors = 1036 признаков**.

Целевые границы:

- absorption: 290–420 nm;
- log extinction ≥ 3.80;
- photochem efficiency ≥ 0.25;
- log10 half-life(s) ≥ 3.56;
- logKp ≤ −5.00;
- sensitization probability ≤ 0.50;
- irritation probability ≤ 0.50.

## 3. Почему по умолчанию 20 RL-шагов, а не 60

В старом запуске был один seed: `60 × 64 × 7 = 26 880` molecule-rows в RL scoring.

Если механически повторить 60 шагов на трёх seeds, получится:

`60 × 64 × 7 × 3 = 80 640`, что существенно выше рекомендованного в задании лимита 30 000 молекул для основного обучения.

Поэтому в этом notebook по умолчанию:

`20 × 64 × 7 × 3 = 26 880`.

Так мы получаем **3 независимых повтора при том же порядке общего training budget**, что и прежний одно-seed эксперимент. Финальный sampling не меняет веса и анализируется отдельно.

In [ ]:
# ===================== НАСТРОЙКИ ЭКСПЕРИМЕНТА =====================
SEEDS = [42, 101, 2024]
STEPS_PER_PROFILE = 20
BATCH_SIZE = 64
SAMPLE_PER_PROFILE = 500

RUN_SETUP = True
RUN_CHECK = True
RUN_SMOKE = True
RUN_MAIN_M1 = True
LOAD_B0_REFERENCE = True
RUN_ABLATIONS = True

import shutil, subprocess, os, sys

# Для полного M1 эксперимента GPU обязателен практически, хотя код технически умеет CPU.
HAS_NVIDIA = shutil.which("nvidia-smi") is not None
if HAS_NVIDIA:
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True
    ).stdout.strip()
    print("GPU:", gpu_info)
else:
    print("GPU не обнаружен.")

if RUN_MAIN_M1 and not HAS_NVIDIA:
    raise RuntimeError(
        "Для полного 3-seed M1 эксперимента включите GPU в Colab: "
        "Runtime / Среда выполнения -> Change runtime type / Сменить среду выполнения -> T4 GPU (или L4). "
        "После смены runtime выполните notebook заново сверху."
    )

PROCESSOR = "cu126" if HAS_NVIDIA else "cpu"
DEVICE = "cuda:0" if HAS_NVIDIA else "cpu"

planned_training_rows = len(SEEDS) * 7 * STEPS_PER_PROFILE * BATCH_SIZE
print("SEEDS:", SEEDS)
print("PROCESSOR:", PROCESSOR, "DEVICE:", DEVICE)
print("Planned RL scoring rows:", f"{planned_training_rows:,}")
if planned_training_rows > 30000:
    print("WARNING: превышен рекомендованный лимит 30,000 training molecule-rows.")
else:
    print("OK: training budget <= 30,000 molecule-rows.")


## 4. Разворачиваем воспроизводимую сборку

Сборка встроена прямо в notebook. Она содержит только код и manifests; тяжёлые evaluator/oracle joblib скачиваются **по pinned Case commit** и проверяются по Git blob SHA/размеру.

Это защищает эксперимент от ситуации, когда содержимое `main` изменилось между запусками.

In [ ]:
import base64, zipfile, pathlib, shutil, os
ROOT = pathlib.Path("/content/REINVENT4_MOST_3SEEDS_ORACLE_V7")
ARCHIVE = pathlib.Path("/content/REINVENT4_MOST_3SEEDS_ORACLE_V7.zip")

_payload = "".join([
'...'
])
ARCHIVE.write_bytes(base64.b64decode(_payload))
if ROOT.exists():
    shutil.rmtree(ROOT)
with zipfile.ZipFile(ARCHIVE) as z:
    z.extractall("/content")
print("Project:", ROOT)
print((ROOT / "README_RU.md").read_text(encoding="utf-8")[:1800])

## 5. Setup: REINVENT4 + evaluators + oracle

Setup выполняет:

1. установку pinned REINVENT4;
2. создание отдельных environments для engine и property models;
3. загрузку **7 evaluator** и **7 oracle** моделей из Case;
4. загрузку D_A/D_B для Applicability Domain;
5. проверку 1036-feature representation;
6. реальную загрузку evaluator и oracle joblib и тестовый prediction.

Если setup падает, ячейка ниже сохраняет полный лог и показывает хвост ошибки вместо непрозрачного `CalledProcessError`. При первой неудаче она удаляет только воспроизводимые virtualenv и делает одну чистую повторную попытку.

In [ ]:
import subprocess, pathlib, shutil, os, sys, time

ROOT = pathlib.Path("/content/REINVENT4_MOST_3SEEDS_ORACLE_V7")
LOG_DIR = ROOT / "colab_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


def _tail_text(path, n=160):
    p = pathlib.Path(path)
    if not p.exists():
        return f"<log not found: {p}>"
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])


def _resource_status():
    parts = []
    try:
        info = {}
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            k, v = line.split(':', 1)
            info[k] = v.strip()
        avail = int(info['MemAvailable'].split()[0]) / 1024 / 1024
        total = int(info['MemTotal'].split()[0]) / 1024 / 1024
        parts.append(f"RAM {avail:.1f}/{total:.1f} GiB available")
    except Exception:
        pass
    try:
        du = shutil.disk_usage('/content')
        parts.append(f"disk {du.free/1024**3:.1f} GiB free")
    except Exception:
        pass
    if shutil.which('nvidia-smi'):
        try:
            x = subprocess.run(
                ['nvidia-smi','--query-gpu=memory.used,memory.total','--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=5
            )
            if x.returncode == 0:
                parts.append('GPU MiB ' + x.stdout.strip().replace('\n','; '))
        except Exception:
            pass
    return ' | '.join(parts)


def run_logged(cmd, log_name, cwd=ROOT, check=True, heartbeat_seconds=30):
    """Run a heavy command without streaming thousands of lines into Colab UI."""
    log_path = LOG_DIR / log_name
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))
    print("Full log:", log_path)

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.setdefault('PYTHONWARNINGS', 'ignore::DeprecationWarning')

    started = time.time()
    with log_path.open('w', encoding='utf-8', buffering=1) as log:
        p = subprocess.Popen(
            cmd, cwd=str(cwd), stdout=log, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', env=env
        )
        last = -heartbeat_seconds
        while True:
            rc = p.poll()
            elapsed = time.time() - started
            if elapsed - last >= heartbeat_seconds:
                print(f"[{elapsed/60:.1f} min] running | {_resource_status()}", flush=True)
                last = elapsed
            if rc is not None:
                break
            time.sleep(2)

    if rc != 0:
        print(f"FAILED, return code={rc}")
        print("\n--- REAL ERROR: LAST LOG LINES ---")
        print(_tail_text(log_path, 180))
        if check:
            raise RuntimeError(f"Command failed. Read the REAL ERROR block above. Full log: {log_path}")
        return rc

    print(f"DONE in {(time.time()-started)/60:.1f} min")
    print("--- log tail ---")
    print(_tail_text(log_path, 25))
    return rc


def ensure_uv_python312():
    """Use a known-compatible Python 3.12 for REINVENT/PyTorch setup."""
    uv = shutil.which('uv')
    if uv is None:
        print('Installing uv bootstrap...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        uv = shutil.which('uv')
    if uv is None:
        raise RuntimeError('uv installation succeeded but executable was not found')

    # Idempotent: downloads Python 3.12 only if it is not already present.
    subprocess.run([uv, 'python', 'install', '3.12'], check=True)
    r = subprocess.run([uv, 'python', 'find', '3.12'], capture_output=True, text=True, check=True)
    py = r.stdout.strip().splitlines()[-1].strip()
    if not pathlib.Path(py).is_file():
        raise RuntimeError(f'uv returned invalid Python 3.12 path: {py}')
    print('Pinned setup Python:', py)
    subprocess.run([py, '-c', 'import sys; print(sys.version)'], check=True)
    return py


RUNNER_PY = ensure_uv_python312()


def setup_with_one_clean_retry():
    cmd = [RUNNER_PY, 'run.py', 'setup', '--processor', PROCESSOR]

    # Cell 6 recreates ROOT, but remove only reproducible env/cache fragments before setup.
    def clean_reproducible_partial_state():
        for name in ['.venv-engine', '.venv-evaluators']:
            p = ROOT / name
            if p.exists():
                shutil.rmtree(p, ignore_errors=True)
        for p in ROOT.rglob('*.partial'):
            p.unlink(missing_ok=True)

    try:
        run_logged(cmd, 'setup_attempt1.log')
        return
    except Exception as first:
        print('\nFirst setup attempt failed.')
        print('The exact cause is above and in setup_attempt1.log.')
        print('Cleaning only reproducible partial environments and retrying once...')
        clean_reproducible_partial_state()
        try:
            run_logged(cmd, 'setup_attempt2.log')
            return
        except Exception:
            print('\n========== SETUP FAILED TWICE =========')
            print('\n--- attempt 1 tail ---')
            print(_tail_text(LOG_DIR / 'setup_attempt1.log', 120))
            print('\n--- attempt 2 tail ---')
            print(_tail_text(LOG_DIR / 'setup_attempt2.log', 180))
            raise RuntimeError(
                'Setup failed twice. The actual package/network/model error is printed immediately above; '
                'do not use the outer RuntimeError as the diagnosis.'
            ) from first


if RUN_SETUP:
    setup_with_one_clean_retry()
else:
    print('RUN_SETUP=False — using an already prepared environment.')


### Если setup всё-таки остановился

Не копируйте только внешний `RuntimeError`: он является оболочкой. Реальная причина автоматически печатается под заголовком **REAL ERROR: LAST LOG LINES**.

Для ручной диагностики можно выполнить:

```python
print(_tail_text(LOG_DIR / "setup_attempt2.log", 200))
```

Наиболее важные строки — последние `ERROR`, `Traceback`, `No matching distribution`, `HTTP Error`, `Killed` или `CUDA` сообщения.


## 6. Unit tests и preflight

`check` проверяет, что:

- Case commit зафиксирован;
- ровно 7 evaluator и 7 oracle описаны в manifest;
- reward config не содержит oracle;
- признаки и целевые границы согласованы;
- Pareto sorting и Eyring proxy работают;
- evaluator/oracle stack можно загрузить.

После этого короткий smoke-run проверяет реальный цикл REINVENT → evaluator reward → checkpoint → final oracle post-score.

In [ ]:
# Используем тот же pinned Python 3.12 orchestration process, что и на setup.
if RUN_CHECK:
    run_logged([RUNNER_PY, "run.py", "check", "--seed", str(SEEDS[0])], "check.log")
if RUN_SMOKE:
    run_logged([
        RUNNER_PY, "run.py", "smoke",
        "--steps", "2", "--batch-size", "16",
        "--device", DEVICE, "--seed", str(SEEDS[0])
    ], "smoke.log")


## 7. Основной M1 эксперимент: 3 seeds × 7 priorities

Для каждого seed создаётся семь копий одного и того же REINVENT prior. Отличается только priority профиля. Каждая молекула внутри конкретного batch оценивается всеми семью surrogate evaluators.

**Oracle не участвует в RL.** После завершения обучения каждого агента выполняется финальный sampling, surrogate scoring, а затем отдельный oracle process.

При настройках по умолчанию обучается 21 agent.

In [ ]:
if RUN_MAIN_M1:
    cmd = [
        RUNNER_PY, "run.py", "experiment",
        "--steps", str(STEPS_PER_PROFILE),
        "--batch-size", str(BATCH_SIZE),
        "--n", str(SAMPLE_PER_PROFILE),
        "--device", DEVICE,
        "--seeds", *map(str, SEEDS),
    ]
    run_logged(cmd, "experiment_3seeds.log")
else:
    print("RUN_MAIN_M1=False — основной M1 run пропущен.")


## 8. Находим результаты и загружаем таблицы

Ключевая таблица — `seed_metrics.csv`. Она позволяет сравнить три независимых запуска. `seed_metrics_mean_std.csv` содержит агрегированную статистику.

Важно различать:

- `JSR_Surrogate_raw` — joint pass по тем моделям, которые направляли RL;
- `JSR_Oracle` — joint pass по независимым oracle-моделям;
- `JSR_Oracle_reliable_AD_SAS` — более консервативный вариант: oracle pass + оба AD + SAS.

In [ ]:
import json, pandas as pd, numpy as np
from pathlib import Path

latest_file = ROOT / "runs" / "latest_3seed_oracle_experiment.json"
if not latest_file.exists():
    raise FileNotFoundError("Нет результата M1. Сначала выполните основной эксперимент.")
EXP = Path(json.loads(latest_file.read_text(encoding="utf-8"))["experiment"])
AGG = EXP / "aggregate"
print("Experiment:", EXP)

seed_metrics = pd.read_csv(AGG / "seed_metrics.csv")
mean_std = pd.read_csv(AGG / "seed_metrics_mean_std.csv")
profile_summary = pd.read_csv(AGG / "profile_summary.csv")
summary = json.loads((AGG / "summary.json").read_text(encoding="utf-8"))

display(seed_metrics)
display(mean_std.T.rename(columns={0:"value"}))
print(json.dumps({k:v for k,v in summary.items() if k not in {"per_seed","mean_std"}}, indent=2, ensure_ascii=False))

## 9. Проверка: действительно ли у каждого seed достаточно уникальных молекул

В официальном результате желательно иметь не менее 1000 новых структур. Здесь мы проверяем это отдельно для каждого seed.

Для честного сравнения с baseline B0 агрегатор также создаёт `fair_eval_candidates.csv` размером до 1000 уникальных молекул на seed.

In [ ]:
for _, r in seed_metrics.iterrows():
    seed = int(r["seed"])
    n_unique = int(r["N_unique"])
    fair_n = int(r["Fair_Eval_N"])
    status = "OK" if n_unique >= 1000 else "WARNING"
    print(f"seed={seed}: unique={n_unique:,}; fair_eval={fair_n:,} -> {status}")
print("Если unique < 1000, увеличьте SAMPLE_PER_PROFILE и повторите только основной эксперимент.")

## 10. Визуализация JSR по трём seeds

Если `JSR_Surrogate` высок, а `JSR_Oracle` заметно ниже, это признак surrogate exploitation / Goodhart effect. Если улучшение переносится на oracle и стабильно между seeds, аргумент в пользу реального эффекта M1 существенно сильнее.

In [ ]:
import matplotlib.pyplot as plt
plot_df = seed_metrics.set_index("seed")[["JSR_Surrogate_raw", "JSR_Oracle", "JSR_Oracle_reliable_AD_SAS"]]
ax = plot_df.plot(kind="bar", figsize=(10,5))
ax.set_ylabel("Rate")
ax.set_title("M1: surrogate vs independent oracle JSR by seed")
ax.set_ylim(0, max(0.05, float(plot_df.max().max())*1.25))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Mean ± std — числа для защиты

Для трёх seeds нужно сообщать не одно число, а среднее и разброс. Это позволяет показать устойчивость результата к случайности sampling/RL.

In [ ]:
def ms(metric):
    vals = pd.to_numeric(seed_metrics[metric], errors="coerce").dropna().to_numpy(float)
    return float(vals.mean()), float(vals.std(ddof=0))

for metric in [
    "Validity", "Uniqueness_across_profiles", "Novelty", "Internal_Diversity",
    "JSR_Surrogate_raw", "JSR_Oracle", "JSR_Oracle_reliable_AD_SAS",
    "AD_Both_Rate", "SAScore_Pass_Rate", "Robust_Surrogate_Rate"
]:
    m, s = ms(metric)
    print(f"{metric:34s}: {m:.4f} ± {s:.4f}  ({m*100:.2f}% ± {s*100:.2f}% if rate)")

## 12. B0 baseline из того же pinned Case

Для сравнения используем B0, уже реализованный в Case с **теми же seeds 42 / 101 / 2024** и тем же независимым oracle stack.

Notebook загружает только опубликованные B0 results из pinned commit `aa9c32fb26a39e518e420bb333298a20d59cbd94` — не из плавающего `main`.

Это даёт контрольную точку. Для максимально строгого paper-level сравнения B0 и M1 следует запускать в одном hardware/runtime budget; здесь дополнительно используется `Fair_JSR_Oracle` на выборке до 1000 уникальных M1-кандидатов на seed.

In [ ]:
import urllib.request, time
B0_DIR = Path("/content/Case_B0_reference")
B0_DIR.mkdir(exist_ok=True)
CASE_COMMIT = "aa9c32fb26a39e518e420bb333298a20d59cbd94"

def download_small(rel):
    url = f"https://raw.githubusercontent.com/suharevalexey/Case/{CASE_COMMIT}/{rel}"
    dst = B0_DIR / Path(rel).name
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    last = None
    for i in range(3):
        try:
            urllib.request.urlretrieve(url, dst)
            return dst
        except Exception as e:
            last = e; time.sleep(2*(i+1))
    raise RuntimeError(f"Cannot download {url}: {last}")

b0 = None
if LOAD_B0_REFERENCE:
    b0_path = download_small("results/metrics_summary_B0.csv")
    b0 = pd.read_csv(b0_path)
    display(b0)
else:
    print("LOAD_B0_REFERENCE=False")

## 13. B0 vs M1: независимый Oracle JSR

Основная проверка исследовательской гипотезы должна опираться на **oracle**, а не на reward score.

Для M1 используем `Fair_JSR_Oracle` — одинаковый верхний размер evaluation subset (до 1000 уникальных кандидатов на seed). Для B0 в Case каждая строка seed уже соответствует 1000 уникальным финальным кандидатам.

In [ ]:
comparison_rows = []
if b0 is not None:
    for seed in SEEDS:
        b0r = b0[(b0["method"] == "B0") & (pd.to_numeric(b0["seed"], errors="coerce") == seed)]
        m1r = seed_metrics[seed_metrics["seed"] == seed]
        if len(b0r) and len(m1r):
            comparison_rows.append({
                "seed": seed,
                "B0_JSR_Oracle": float(b0r.iloc[0]["JSR_Oracle (Joint Success Rate)"]),
                "M1_Fair_JSR_Oracle": float(m1r.iloc[0]["Fair_JSR_Oracle"]),
                "M1_minus_B0": float(m1r.iloc[0]["Fair_JSR_Oracle"]) - float(b0r.iloc[0]["JSR_Oracle (Joint Success Rate)"]),
                "B0_Novelty": float(b0r.iloc[0]["Novelty"]),
                "M1_Novelty": float(m1r.iloc[0]["Novelty"]),
                "B0_Diversity": float(b0r.iloc[0]["Internal_Diversity"]),
                "M1_Diversity": float(m1r.iloc[0]["Internal_Diversity"]),
                "B0_AD_Both": float(b0r.iloc[0]["AD_Fraction_in_Both_AD"]),
                "M1_AD_Both": float(m1r.iloc[0]["AD_Both_Rate"]),
            })
comparison = pd.DataFrame(comparison_rows)
display(comparison)
if len(comparison):
    print("Mean M1-B0 Oracle JSR delta:", comparison["M1_minus_B0"].mean())

## 14. Абляция 1 — AD/SAS gate ON vs OFF

Эта проверка отвечает на вопрос: **не появляется ли formal joint success в основном за счёт ненадёжной экстраполяции?**

- OFF: считаем только независимый oracle pass;
- ON: дополнительно требуем оба AD и SAScore ≤ 5.

Если после включения gate успех резко исчезает, метод нашёл кандидатов преимущественно в менее надёжной области.

In [ ]:
global_df = pd.read_csv(AGG / "all_unique_candidates_global.csv")

def boolcol(df, name):
    s = df[name] if name in df.columns else pd.Series(False, index=df.index)
    if s.dtype == bool: return s.fillna(False)
    return s.fillna(False).astype(str).str.lower().isin(["true","1","yes"])

oracle_pass = boolcol(global_df, "oracle_pass_all")
in_ad = boolcol(global_df, "inside_both_ad")
sa = boolcol(global_df, "sascore_pass")

ablation_ad = pd.DataFrame([
    {"variant":"AD/SAS OFF", "N":len(global_df), "Oracle_success_rate":float(oracle_pass.mean()), "N_success":int(oracle_pass.sum())},
    {"variant":"AD/SAS ON", "N":len(global_df), "Oracle_success_rate":float((oracle_pass & in_ad & sa).mean()), "N_success":int((oracle_pass & in_ad & sa).sum())},
])
display(ablation_ad)

## 15. Абляция 2 — Pareto vs equal-weight scalarization

Здесь oracle **не используется для выбора**. На одном и том же feasible пуле:

- Pareto выбирает rank 0;
- scalarization ранжирует по среднему семи surrogate utilities с равными весами;
- затем независимый oracle только измеряет JSR выбранных наборов.

Размер scalarized top-K делаем равным размеру Pareto-front, чтобы сравнение не выигрывалось просто за счёт количества кандидатов.

In [ ]:
utility_cols = [c for c in global_df.columns if c.startswith("utility_group_")]
feasible = global_df[in_ad & sa].copy()
pareto_set = feasible[pd.to_numeric(feasible["pareto_rank_global"], errors="coerce") == 0].copy()
feasible["equal_weight_score"] = feasible[utility_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
k = len(pareto_set)
scalar_set = feasible.sort_values("equal_weight_score", ascending=False).head(k).copy()

ablation_merge = pd.DataFrame([
    {"variant":"Pareto rank 0", "N_selected":len(pareto_set), "Oracle_JSR":float(boolcol(pareto_set,"oracle_pass_all").mean()) if len(pareto_set) else np.nan},
    {"variant":"Equal-weight scalarization top-K", "N_selected":len(scalar_set), "Oracle_JSR":float(boolcol(scalar_set,"oracle_pass_all").mean()) if len(scalar_set) else np.nan},
])
display(ablation_merge)

## 16. Абляция 3 — leave-one-priority-out

Дополнительная проверка: по очереди убираем результаты одного priority-профиля и смотрим, как меняется число уникальных структур и независимый Oracle JSR.

Это показывает, действительно ли семь направлений поиска дополняют друг друга, или некоторые почти полностью дублируются.

In [ ]:
rows = pd.read_csv(AGG / "all_profile_rows.csv")
priority_values = sorted(rows["source_profile"].dropna().unique())
loo = []
for p in priority_values:
    sub = rows[rows["source_profile"] != p].copy()
    sub = sub[sub["canonical_smiles"].notna()].copy()
    sub = sub.drop_duplicates("canonical_smiles")
    loo.append({
        "removed_priority": p,
        "N_unique_remaining": len(sub),
        "Oracle_JSR_remaining": float(boolcol(sub,"oracle_pass_all").mean()) if len(sub) else np.nan,
        "Oracle_reliable_rate_remaining": float((boolcol(sub,"oracle_pass_all") & boolcol(sub,"inside_both_ad") & boolcol(sub,"sascore_pass")).mean()) if len(sub) else np.nan,
    })
loo_df = pd.DataFrame(loo).sort_values("N_unique_remaining")
display(loo_df)

## 17. Failure analysis — где surrogate и oracle расходятся

Если evaluator говорит `pass`, а независимый oracle — `fail`, это особенно важный failure mode: генератор мог оптимизировать особенность surrogate, которая не переносится на независимую модель.

Ниже считаем disagreement для каждого из семи свойств.

In [ ]:
property_keys = [
    "group_A_absorption_max_nm", "group_A_log_extinction",
    "group_A_photochem_efficiency", "group_A_log_half_life",
    "group_B_log_kp", "group_B_skin_sensitization", "group_B_skin_irritation"
]
fail_rows=[]
for key in property_keys:
    sp = boolcol(global_df, f"pass_{key}")
    op = boolcol(global_df, f"oracle_pass_{key}")
    fail_rows.append({
        "property": key,
        "surrogate_pass_rate": float(sp.mean()),
        "oracle_pass_rate": float(op.mean()),
        "disagreement_rate": float((sp != op).mean()),
        "surrogate_pass_oracle_fail": int((sp & ~op).sum()),
    })
failure = pd.DataFrame(fail_rows).sort_values("disagreement_rate", ascending=False)
display(failure)

## 18. Экспорт generated.csv и таблиц для отчёта

Для `generated.csv` объединяем fair evaluation subset трёх seeds. При нормальном запуске это до **3000 уникальных seed-level записей** (1000 на seed), с:

- SMILES;
- method/seed/profile;
- 7 surrogate predictions;
- uncertainty;
- 7 oracle predictions;
- novelty;
- SAScore;
- distances to D_A / D_B;
- AD status;
- surrogate/oracle pass flags.

In [ ]:
EXPORT = Path("/content/REINVENT4_MOST_3SEEDS_FINAL_EXPORT")
EXPORT.mkdir(exist_ok=True)

fair_frames=[]
for seed in SEEDS:
    p = AGG / f"seed_{seed}" / "fair_eval_candidates.csv"
    if p.exists():
        x = pd.read_csv(p)
        x["method_id"] = "M1_REINVENT4_PARETO"
        x["evaluation_seed"] = seed
        fair_frames.append(x)
if not fair_frames:
    raise RuntimeError("Fair evaluation files not found")
generated = pd.concat(fair_frames, ignore_index=True)
generated.to_csv(EXPORT / "generated_M1_3seeds.csv", index=False)
seed_metrics.to_csv(EXPORT / "M1_seed_metrics.csv", index=False)
mean_std.to_csv(EXPORT / "M1_mean_std.csv", index=False)
profile_summary.to_csv(EXPORT / "M1_profile_summary.csv", index=False)
comparison.to_csv(EXPORT / "B0_vs_M1.csv", index=False)
ablation_ad.to_csv(EXPORT / "ablation_AD_SAS.csv", index=False)
ablation_merge.to_csv(EXPORT / "ablation_Pareto_vs_scalarization.csv", index=False)
loo_df.to_csv(EXPORT / "ablation_leave_one_priority_out.csv", index=False)
failure.to_csv(EXPORT / "failure_evaluator_vs_oracle.csv", index=False)

print("generated rows:", len(generated))
print("unique canonical SMILES globally in export:", generated["canonical_smiles"].nunique())
print("Export dir:", EXPORT)

## 19. Автоматический текст для защиты

Этот блок формирует формулировку только из реально полученных чисел. Он не подставляет старые результаты v6.

In [ ]:
m_jsr, s_jsr = ms("Fair_JSR_Oracle")
m_surr, s_surr = ms("JSR_Surrogate_raw")
m_nov, s_nov = ms("Novelty")
m_div, s_div = ms("Internal_Diversity")
m_ad, s_ad = ms("AD_Both_Rate")

print(f"""
Основной M1-эксперимент был повторён на трёх независимых seeds: {SEEDS}.
Для каждого seed обучалось семь REINVENT4-агентов с разными property priorities,
при этом reward формировался только surrogate evaluators. Независимые oracle-модели
не использовались при обучении и применялись только к финальным кандидатам.

Средний surrogate JSR составил {m_surr*100:.2f}% ± {s_surr*100:.2f}%, а независимый
fair Oracle JSR — {m_jsr*100:.2f}% ± {s_jsr*100:.2f}%.
Novelty: {m_nov*100:.2f}% ± {s_nov*100:.2f}%; internal diversity: {m_div:.4f} ± {s_div:.4f};
доля кандидатов внутри обоих AD: {m_ad*100:.2f}% ± {s_ad*100:.2f}%.
""")

if len(comparison):
    delta = comparison["M1_minus_B0"].mean()
    print(f"Средняя разница M1−B0 по независимому Oracle JSR: {delta*100:+.2f} процентных пункта.")

## 20. Архивируем результаты

В архив входят итоговые CSV анализа. Тяжёлые agent checkpoints уже сохраняются отдельным архивом самим M1-runner.

In [ ]:
import shutil
export_zip = shutil.make_archive("/content/REINVENT4_MOST_3SEEDS_FINAL_EXPORT", "zip", EXPORT)
print("Analysis export:", export_zip)
print("M1 results archive:", "/content/REINVENT4_MOST_3SEEDS_ORACLE_RESULTS.zip")
print("M1 checkpoints archive:", "/content/REINVENT4_MOST_3SEEDS_ORACLE_CHECKPOINTS.zip")

try:
    from google.colab import files
    print("В Colab можно скачать так:")
    print('files.download("/content/REINVENT4_MOST_3SEEDS_FINAL_EXPORT.zip")')
except Exception:
    pass

## 21. Как интерпретировать результат

**Хороший сценарий:** M1 имеет выше `JSR_Oracle`, чем B0, эффект повторяется на всех/большинстве seeds, novelty/diversity не коллапсируют, а AD pass остаётся приемлемым.

**Reward hacking:** `JSR_Surrogate` сильно растёт, но `JSR_Oracle` не растёт или падает.

**Domain exploitation:** Oracle JSR выглядит высоким только при AD/SAS OFF и резко исчезает после gate.

**Нет улучшения над B0:** это не «сломанный проект». Это корректный отрицательный экспериментальный результат: сложный RL/Pareto метод не доказал преимущество над простым baseline при заданном бюджете.

**Ограничение:** oracle всё ещё является ML-моделью, а не лабораторным экспериментом. Он обеспечивает независимую вычислительную оценку, но не доказывает реальную безопасность, SPF, фотостабильность или MOST storage enthalpy.